## Análise resultados do `resultados/comparativo.csv`

Métricas principais: fragmentação interna, fragmentação externa, operações por alocação, taxa de falha e tempo de execução

Workloads: workload 1(Alocações sequenciais de tamanhos variados, sem liberação — mede uso puro de espaço e
fragmentação interna), workload 2 (Alocações e liberações intercaladas (pseudoaleatórias, com semente fixa) — mede
fragmentação externa ao longo do tempo), workload 3(Padrão adversarial: alocar N blocos pequenos, liberar blocos alternados, tentar alocar
um bloco grande que caberia na soma dos buracos, mas não em nenhum deles isoladamente)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Preparação dos dados e leitura

In [ ]:
arq = '../resultados/comparativo.csv'
df = pd.read_csv(arq)

In [ ]:
df.head()
print('Linhas:', len(df))
print('Colunas:')
print(df.columns.tolist())

df.info()

df.shape

In [ ]:
col_numericas = ['heap_size', 'operacoes', 'alocacoes_ok', 'falhas', 'maior_livre', 'total_livre', 'frag_interna', 'blocos_examinados', 'tempo_alloc_ns', 'tempo_free_ns']

for coluna in col_numericas:
    if coluna in df.columns:
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

In [ ]:
df.head()

## Métricas derivadas (Frag interna, Frag externa e Taxa de falha de alocação)

In [ ]:
# Fragmentação interna
if {'maior_livre', 'total_livre'}.issubset(df.columns):
    df['frag_externa'] = np.where(df['total_livre'] > 0, 1 - (df['maior_livre'] / df['total_livre']), 0)

#Fragmentação externa
if {'blocos_examinados', 'alocacoes_ok'}.issubset(df.columns):
    df['examinados_por_alocacao'] = np.where( df['alocacoes_ok'] > 0, df['blocos_examinados'] / df['alocacoes_ok'], np.nan)

#Taxa de falha da alocação
if {'falhas', 'operacoes'}.issubset(df.columns):
    df['taxa_falha'] = np.where( df['operacoes'] > 0, 100 * df['falhas'] / df['operacoes'], 0)

df.head()

In [ ]:
df.groupby(['workload', 'estrategia']).agg({
    'alocacoes_ok': 'sum',
    'falhas': 'sum',
    'frag_externa': 'mean',
    'examinados_por_alocacao': 'mean'
}).reset_index()

## Wolkload 1 - alocações sequenciais

In [ ]:
w1 = df[df['workload']] == 'workload1'.copy()
w1.groupby('estrategia').agg({'alocacoes_ok':'sum','frag_interna':'max'}).reset_index()

In [ ]:
plt.figure(figsize=(8, 5))
for estrategia, grupo in w1.groupby('estrategia'):
    plt.bar(estrategia, grupo['frag_interna'].iloc[-1])
plt.xlabel('Estratégia')
plt.ylabel('Fragmentação interna (bytes)')
plt.title('Workload 1 — Fragmentação interna')
plt.show()

## Workload 2 - operações intercaladas

In [ ]:
w2 = df[df['workload'] == 'workload2'].copy()

plt.figure(figsize=(10, 5))
for estrategia, grupo in w2.groupby('estrategia'):
    grupo = grupo.sort_values('operacao')
    plt.plot(grupo['operacao'], grupo['frag_externa'] * 100, label=estrategia)

In [ ]:
plt.xlabel('Operação')
plt.ylabel('Fragmentação externa (%)')
plt.title('Workload 2 — Fragmentação externa ao longo das operações')
plt.legend()
plt.grid(True)
plt.show()

## Workload 3 - padrão adversarial

In [ ]:
w3 = df[df['workload'] == 'workload3'].copy()

w3.groupby('estrategia').agg({'falhas': 'sum', 'frag_externa': 'max', 'maior_livre': 'min'}).reset_index()

## Número de operações por alocação

In [ ]:
custo = df.groupby('estrategia')['examinados_por_alocacao'].mean().sort_values()

plt.figure(figsize=(8, 5))
plt.bar(custo.index, custo.values)
plt.xlabel('Estratégia')
plt.ylabel('Blocos examinados por alocação')
plt.title('Custo médio da busca')
plt.show()

## Tempo de execução

In [ ]:
if 'tempo_alloc_ns' in df.columns:
    tempo = df.groupby('estrategia')['tempo_alloc_ns'].agg(
        p50=lambda x: np.percentile(x.dropna(), 50),
        p95=lambda x: np.percentile(x.dropna(), 95)
    ).reset_index()
    tempo

# mediana (p50) e percentil 95 (p95)

## Tabela final resumida

In [ ]:
resumo = df.groupby('estrategia').agg({
    'alocacoes_ok': 'sum',
    'falhas': 'sum',
    'frag_externa': 'mean',
    'examinados_por_alocacao': 'mean'
}).reset_index()

resumo